##### 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time
import requests
from math import radians, sin, cos, sqrt, atan2
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

In [ ]:
# 없으면 다운로드 필요
# pip install openpyxl
# pip install openpyxl requests

In [ ]:
# 기본 설정
VWORLD_KEY = 'KEYINSERT'

# 파일 경로
APT_FILES = [
    '아파트(매매)_실거래가_1.csv',
    '아파트(매매)_실거래가_2.csv',
    '아파트(매매)_실거래가_3.csv',
]
SUBWAY_FILE  = '서울시_역사.csv'
SCHOOL_FILE  = '한국교육시설안전원_초중등학교위치.csv'
RATE_FILE    = '한국은행_기준금리_및_여수신금리.csv'

##### 데이터 불러오기 & 기본 필터링

In [ ]:
dfs = [pd.read_csv(f, encoding='cp949', skiprows=15) for f in APT_FILES]
df = pd.concat(dfs, ignore_index=True)

# 계약 해제 제거
before = len(df)
df = df[df['해제사유발생일'] == '-'].copy()
print(f"[필터] 해제 제거: {len(df):,}건 ({before-len(df):,}건 삭제)")

# 지하층 제거
before = len(df)
df = df[df['층'] >= 0].copy()
print(f"[필터] 지하층 제거: {len(df):,}건 ({before-len(df):,}건 삭제)")

# 괄호+숫자만으로 된 단지명 제거
before = len(df)
df = df[~df['단지명'].str.match(r'^\([\d\-]+\)$', na=False)].copy()
print(f"[필터] 괄호 단지명 제거: {len(df):,}건 ({before-len(df):,}건 삭제)")


In [ ]:
# 단지명 필터링
def clean_apt_name(name: str) -> str:
    # 1. 쉼표 제거
    name = re.sub(r',+', '', name)
     # 2. 괄호 안에 하이픈 포함된 패턴이면 괄호째 삭제 (890-49, 1동-7동)
    name = re.sub(r'\([^)]*-[^)]*\)', '', name)
    # 2-1. 괄호 안 숫자만 있으면 괄호째 삭제 (336), (963)
    name = re.sub(r'\(\d+\)', '', name)
    # 3. 괄호 안 숫자~숫자 패턴이면 괄호째 삭제 (208~211동)
    name = re.sub(r'\([^)]*~[^)]*\)', '', name)
    # 4. 숫자~숫자 패턴 제거 (1동~8동)
    name = re.sub(r'\d+동?~\d+동?', '', name)
    # 5. 동 번호 제거 (1동, 901동)
    name = re.sub(r'\d+동', '', name)
    # 6. 차단지 → 단지로 변환 (5차단지 → 5단지)
    name = re.sub(r'(\d+)차단지', r'\1단지', name)
    # 7. 남은 괄호 제거
    name = re.sub(r'[()]', '', name)
    # 8. 한글 뒤 숫자로 끝나는 경우만 차 추가 (우성1 → 우성1차)
    name = re.sub(r'([가-힣])(\d+)$',
                  lambda m: m.group(1) + m.group(2) + '차', name)
    return name.strip()

df['단지명'] = df['단지명'].apply(clean_apt_name)
print(f"[필터] 단지명 필터링 완료. unique 단지: {df['단지명'].nunique():,}개")

##### VWorld 지오코딩

In [ ]:
# 도로명 전체 주소 생성
df['도로명_전체'] = '서울특별시 ' + df['시군구'].str.extract(r'서울특별시\s+(\S+구)')[0] + ' ' + df['도로명']

# 단지 unique 추출 (API 호출 최소화)
단지_master = df[['단지명', '도로명_전체']].drop_duplicates().reset_index(drop=True)
print(f"[지오코딩] 대상 단지: {len(단지_master):,}개")

def vworld_geocode(address: str) -> tuple:
    """VWorld API로 주소 → (위도, 경도) 반환. 실패 시 (None, None)"""
    url = 'https://api.vworld.kr/req/address'
    params = {
        'service': 'address',
        'request': 'getcoord',
        'version': '2.0',
        'crs': 'epsg:4326',
        'address': address,
        'refine': 'true',
        'simple': 'false',
        'format': 'json',
        'type': 'road',
        'key': VWORLD_KEY,
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json()
        if data['response']['status'] == 'OK':
            pt = data['response']['result']['point']
            return float(pt['y']), float(pt['x'])  # 위도, 경도
    except Exception:
        pass
    return None, None

# API 키 사전 검증
test_lat, test_lng = vworld_geocode('서울특별시 강남구 테헤란로 152')
if test_lat is None:
    raise RuntimeError("API 연결 실패: VWorld API 키를 확인하세요. (설정 셀의 VWORLD_KEY)")
print("[검증] API 연결 정상")

lats, lngs = [], []
for i, row in 단지_master.iterrows():
    lat, lng = vworld_geocode(row['도로명_전체'])
    lats.append(lat)
    lngs.append(lng)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(단지_master)} 완료")
    time.sleep(0.1)

단지_master['위도'] = lats
단지_master['경도'] = lngs

success = 단지_master['위도'].notna().sum()
print(f"[지오코딩] 성공: {success:,} / {len(단지_master):,} ({success/len(단지_master)*100:.1f}%)")


In [ ]:
# Haversine 거리 함수
def haversine(lat1, lng1, lat2, lng2):
    """두 위경도 좌표 간 거리(미터) 반환"""
    R = 6371000
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def nearest_and_count(apt_lat, apt_lng, target_df, lat_col, lng_col, name_col,
                       extra_col=None, radius_m=1000):
    """
    단지 좌표에서 target_df의 각 지점까지 haversine 거리 계산.
    반환: (최근접_이름, 최근접_extra, 최근접_거리m, radius_m내_개수)
    """
    if pd.isna(apt_lat) or pd.isna(apt_lng):
        return None, None, None, None
    dists = target_df.apply(
        lambda r: haversine(apt_lat, apt_lng, r[lat_col], r[lng_col]), axis=1
    )
    idx_min = dists.idxmin()
    nearest_name  = target_df.loc[idx_min, name_col]
    nearest_extra = target_df.loc[idx_min, extra_col] if extra_col else None
    nearest_dist  = round(dists[idx_min])
    count_in_r    = (dists <= radius_m).sum()
    return nearest_name, nearest_extra, nearest_dist, count_in_r

In [ ]:
# 지하철역 거리 계산
subway = pd.read_csv(SUBWAY_FILE, encoding='cp949')
print(f"[지하철] {len(subway):,}개 역 로드")

res_subway = 단지_master.apply(
    lambda r: nearest_and_count(
        r['위도'], r['경도'],
        subway, '위도', '경도', '역사명', extra_col='호선'
    ), axis=1, result_type='expand'
)
res_subway.columns = ['최근접_역명', '최근접_역_호선', '최근접_역_거리(m)', '_subway_count']
단지_master = pd.concat([단지_master, res_subway[['최근접_역명','최근접_역_호선','최근접_역_거리(m)']]], axis=1)
print("[지하철] 거리 계산 완료")

In [ ]:
# 학교 거리/수 계산
school_raw = pd.read_csv(SCHOOL_FILE, encoding='utf-8')

# 서울 + 운영 중인 학교만 필터
school_raw = school_raw[
    school_raw['소재지지번주소'].str.startswith('서울', na=False) &
    (school_raw['운영상태'] == '운영')
].copy()
print(f"[학교] 서울 학교 필터 후: {len(school_raw):,}개")

# 학교급별 분리
elem   = school_raw[school_raw['학교급구분'] == '초등학교'].reset_index(drop=True)
middle = school_raw[school_raw['학교급구분'] == '중학교'].reset_index(drop=True)
high   = school_raw[school_raw['학교급구분'] == '고등학교'].reset_index(drop=True)
print(f"  초등: {len(elem)}개 / 중학: {len(middle)}개 / 고등: {len(high)}개")

def calc_school_features(apt_row, school_df, prefix):
    name, _, dist, count = nearest_and_count(
        apt_row['위도'], apt_row['경도'],
        school_df, '위도', '경도', '학교명'
    )
    return pd.Series({
        f'최근접_{prefix}': name,
        f'최근접_{prefix}_거리(m)': dist,
        f'{prefix}_수': count,
    })

for school_df, prefix in [(elem, '초등학교'), (middle, '중학교'), (high, '고등학교')]:
    result = 단지_master.apply(lambda r: calc_school_features(r, school_df, prefix), axis=1)
    단지_master = pd.concat([단지_master, result], axis=1)
    print(f"[학교] {prefix} 거리 계산 완료")

In [ ]:
# 좌표/거리 변수 결합
df['도로명_전체'] = '서울특별시 ' + df['시군구'].str.extract(r'서울특별시\s+(\S+구)')[0] + ' ' + df['도로명']
df = df.merge(단지_master, on=['단지명', '도로명_전체'], how='left')

null_coord = df['위도'].isna().sum()
print(f"[결합] 완료. 좌표 결측: {null_coord}건 ({null_coord/len(df)*100:.2f}%)")

# 동 필터: 숫자 또는 숫자+동 형식만 유지
before = len(df)
df = df[df['동'].astype(str).str.match(r'^\d+동?$')].copy()
print(f"[필터] 동 필터: {len(df):,}건 ({before-len(df):,}건 삭제)")

before = len(df)
df = df[df['위도'].notna()].copy()
print(f"[필터] 좌표 결측 제거: {len(df):,}건 ({before-len(df):,}건 삭제)")


In [ ]:
# 기준금리 매핑 (wide -> long 변환 후 join)
rate_raw = pd.read_csv(RATE_FILE, encoding='utf-8-sig')

# wide → long
rate_cols = [c for c in rate_raw.columns if '/' in c]
rate_long = rate_raw[rate_cols].iloc[0].reset_index()
rate_long.columns = ['연월', '기준금리']
rate_long['계약년월'] = rate_long['연월'].str.replace('/', '').astype(int)
rate_long = rate_long[['계약년월', '기준금리']]

df = df.merge(rate_long, on='계약년월', how='left')

null_rate = df['기준금리'].isna().sum()
print(f"[금리] 매핑 완료. 결측: {null_rate}건")

In [ ]:
# 파생 변수 생성

# 거래금액 정수 변환
df['거래금액_만원'] = (
    df['거래금액(만원)']
    .str.replace(',', '', regex=False)
    .astype(int)
)

# 전용면적 → 평 및 평형대
df['전용면적_평'] = (df['전용면적(㎡)'] / 3.3058).round(2)
df['평형대'] = pd.cut(
    df['전용면적_평'],
    bins=[0, 10, 20, 30, 40, np.inf],
    labels=['10평이하', '10~20평', '20~30평', '30~40평', '40평이상'],
    right=False
)

# 10평 미만 제거
before = len(df)
df = df[df['전용면적_평'] >= 10].copy()
print(f"[필터] 10평 미만 제거: {len(df):,}건 ({before-len(df):,}건 삭제)")

# 건물 연식
df['계약연도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['건물연식'] = df['계약연도'] - df['건축년도']
df['건물연식_구분'] = pd.cut(
    df['건물연식'],
    bins=[0, 5, 10, 20, np.inf],
    labels=['신축(5년이하)', '준신축(5~10년)', '중간(10~20년)', '구축(20년이상)'],
    right=False
)

# 평당가
df['평당가_만원'] = (df['거래금액_만원'] / df['전용면적_평']).round(0)

print("[생성] 파생 변수 완료")

In [ ]:
# 주소 정제 및 집계 변수
df['전체주소'] = df['시군구'] + ' ' + df['도로명']

df['구'] = df['시군구'].str.extract(r'서울특별시\s+(\S+구)')
df['시군구'] = df['시군구'].str.extract(r'서울특별시\s+(\S+구\s+\S+)')
df = df.rename(columns={'번지': '번지수'})

# 구별 / 단지별 평균 평당가
gu_avg  = df.groupby('구')['평당가_만원'].mean().rename('구별_평균평당가')
apt_avg = df.groupby('단지명')['평당가_만원'].mean().rename('단지별_평균평당가')
df = df.merge(gu_avg,  on='구',     how='left')
df = df.merge(apt_avg, on='단지명', how='left')

print("[처리] 주소 정제 및 집계 변수 완료")

##### 최종 컬럼 정리

In [ ]:

FINAL_COLS = [
    # 식별/위치
    '전체주소', '시군구', '번지수', '도로명',
    '단지명', '동', '층',
    # 시간
    '계약년월',
    # 건물 정보
    '건축년도', '건물연식', '건물연식_구분',
    # 면적
    '전용면적(㎡)', '전용면적_평', '평형대',
    # 가격
    '거래금액_만원', '평당가_만원',
    # 거시
    '기준금리',
    # 좌표
    '위도', '경도',
    # 지하철
    '최근접_역명', '최근접_역_호선', '최근접_역_거리(m)',
    # 학교
    '최근접_초등학교', '최근접_초등학교_거리(m)', '초등학교_수',
    '최근접_중학교',   '최근접_중학교_거리(m)',   '중학교_수',
    '최근접_고등학교', '최근접_고등학교_거리(m)', '고등학교_수',
    # 집계
    '구별_평균평당가', '단지별_평균평당가',
]

df = df[FINAL_COLS].sort_values(['시군구', '단지명']).reset_index(drop=True)

print(f"[완료] 최종 데이터: {len(df):,}건 / {len(df.columns)}개 컬럼")
print(df.dtypes)
print("\n[샘플 3건]")
print(df.head(3).to_string())

##### 파일 저장

In [ ]:
# 같은 이름 파일 있으면 이름 변경 후 저장
base     = '아파트_실거래_전처리'
ext      = '.xlsx'
filename = base + ext
counter  = 1
while os.path.exists(filename):
    filename = f'{base}({counter}){ext}'
    counter += 1

df.to_excel(filename, index=False)

# 열 너비 자동 조정
wb = load_workbook(filename)
ws = wb.active
for col in ws.columns:
    max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col)
    ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 2
wb.save(filename)

print(f"[저장] {filename}")